In [1]:
import pandas as pd
import duckdb
import json
import os

# -----------------------------
# 경로
# -----------------------------
train_path = "../data/fs_data/fs_train.parquet"
test_path  = "../data/fs_data/fs_validation.parquet"

REMOVE_LISTS = [
    r'..\config\drop_step1.json',
    r'..\config\drop_step2.json',
]

TRAIN_OUT = "../data/fs_data/fs_train.parquet"
TEST_OUT  = "../data/fs_data/fs_validation.parquet"


# -----------------------------
# 제외 리스트 로드
# -----------------------------
def get_exclusion_list(paths):
    exclude_set = set()
    for path in paths:
        if not os.path.exists(path):
            continue
        with open(path, 'r', encoding='utf-8') as f:
            cfg = json.load(f)
            for key in cfg:
                if isinstance(cfg[key], list):
                    exclude_set.update(cfg[key])
    return list(exclude_set)


# -----------------------------
# 데이터 로드
# -----------------------------
train_df = duckdb.query(f"SELECT * FROM '{train_path}'").df()
test_df  = duckdb.query(f"SELECT * FROM '{test_path}'").df()


# -----------------------------
# 컬럼 제거
# -----------------------------
json_drop = get_exclusion_list(REMOVE_LISTS)

drop_cols = list(set(json_drop))

train_df = train_df.drop(columns=drop_cols, errors='ignore')
test_df  = test_df.drop(columns=drop_cols, errors='ignore')


# -----------------------------
# 저장
# -----------------------------
train_df.to_parquet(TRAIN_OUT, index=False)
test_df.to_parquet(TEST_OUT, index=False)

print("완료")

완료


In [6]:
# 날짜 컷오프!!!!!!!!

import duckdb
import os

# 파일 경로 설정
TRAIN_PATH = "../data/fs_data/fs_train.parquet"
TEST_PATH  = "../data/fs_data/fs_validation.parquet"
CUTOFF_DATE = '2014-04-01'

con = duckdb.connect()

def filter_parquet_by_date(file_path, cutoff_date):
    if not os.path.exists(file_path):
        print(f"❌ 파일을 찾을 수 없습니다: {file_path}")
        return

    print(f"⏳ {file_path} 필터링 중... (기준일: {cutoff_date})")
    
    # 임시 테이블 생성 및 필터링
    # 원본 파일명에 _filtered를 붙여서 저장한 뒤 교체하는 방식이 안전합니다.
    output_path = file_path.replace(".parquet", ".parquet")
    
    query = f"""
    COPY (
        SELECT * FROM read_parquet('{file_path}')
        WHERE date >= '{cutoff_date}'
    ) TO '{output_path}' (FORMAT PARQUET, COMPRESSION 'zstd');
    """
    
    con.execute(query)
    
    # 결과 확인을 위한 카운트 출력
    old_cnt = con.execute(f"SELECT COUNT(*) FROM read_parquet('{file_path}')").fetchone()[0]
    new_cnt = con.execute(f"SELECT COUNT(*) FROM read_parquet('{output_path}')").fetchone()[0]
    
    print(f"✅ 완료: {os.path.basename(file_path)}")
    print(f"   - 기존 행 수: {old_cnt:,}")
    print(f"   - 남은 행 수: {new_cnt:,} (삭제됨: {old_cnt - new_cnt:,})")
    print(f"   - 저장 위치: {output_path}\n")

# 실행
filter_parquet_by_date(TRAIN_PATH, CUTOFF_DATE)
filter_parquet_by_date(TEST_PATH, CUTOFF_DATE)

con.close()

⏳ ../data/fs_data/fs_train.parquet 필터링 중... (기준일: 2014-04-01)
✅ 완료: fs_train.parquet
   - 기존 행 수: 291,988
   - 남은 행 수: 291,988 (삭제됨: 0)
   - 저장 위치: ../data/fs_data/fs_train.parquet

⏳ ../data/fs_data/fs_validation.parquet 필터링 중... (기준일: 2014-04-01)
✅ 완료: fs_validation.parquet
   - 기존 행 수: 669,458
   - 남은 행 수: 669,458 (삭제됨: 0)
   - 저장 위치: ../data/fs_data/fs_validation.parquet



In [ ]:
import duckdb

con = duckdb.connect()
parquet_path = r"..\data\ST4000DM000_v3.parquet"

query = f"""
SELECT 
    failure,
    COUNT(*) as count,
    AVG(smart_9_raw) as avg_age_hours -- 가동 시간(나이) 확인
FROM read_parquet('{parquet_path}')
WHERE date BETWEEN '2014-03-01' AND '2014-03-31'
  AND smart_184_raw IS NULL
GROUP BY failure;
"""

res = con.execute(query).fetchdf()
print(res)

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
C:\Users\joon6\AppData\Local\Temp\ipykernel_25848\194115264.py:4: SyntaxWarning: invalid escape sequence '\d'
  parquet_path = "..\data\ST4000DM000_v3.parquet" # 혹은 원본 파일


   failure  count  avg_age_hours
0        0    360    1702.908333
